# Vanishing and Exploding Gradients

**Goal:** Empirically measure per-layer gradient norms in deep MLPs to demonstrate
vanishing gradients (sigmoid + small init), exploding gradients (large init), and
stable training (ReLU + He/Xavier init). Visualise gradient norm vs. layer depth,
and show gradient clipping bounding the exploding case.

Cross-links: `[[backpropagation]]` · `[[batch-normalization]]`

## Configuration

Device, random seed, and default dtype come from `shared.config.configure()`.
Plots use `matplotlib` with the non-interactive `Agg` backend.

In [1]:
import sys
from pathlib import Path

import matplotlib

matplotlib.use("Agg")  # noqa
import matplotlib.pyplot as plt  # noqa: E402
import torch  # noqa: E402
import torch.nn.functional as F  # noqa: E402


def _find_repo_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "pyproject.toml").exists():
            return p
    return start


REPO_ROOT = _find_repo_root(Path.cwd())
sys.path.insert(0, str(REPO_ROOT))

from shared.config import configure  # noqa: E402

device = configure()
print("running on:", device)

running on: mps


## From Scratch: Deep MLP with Per-Layer Gradient Measurement

We build a 25-layer MLP manually (no `nn.Sequential`) so we can intercept and
record the gradient at each layer's pre-activation during the backward pass.

### Why gradients propagate poorly in deep nets

For a chain `h_L = f_L(f_{L-1}(... f_1(x)))`, the gradient at layer k is:

```
dL/dh_k = dL/dh_L · J_L · J_{L-1} · ... · J_{k+1}
```

If each Jacobian norm `||J_i|| < 1` (typical with sigmoid/tanh), the product
decays exponentially with depth → **vanishing gradients**.
If each `||J_i|| > 1`, the product grows → **exploding gradients**.

In [2]:
import math

N_LAYERS = 25       # deep enough to show exponential decay/growth
WIDTH     = 64      # hidden dimension
BATCH     = 128

torch.manual_seed(0)


def build_weights(n_layers: int, width: int, init_std: float, device: torch.device) -> list[torch.Tensor]:
    """Return a list of weight matrices, each (width, width), with given std."""
    weights = []
    for _ in range(n_layers):
        W = torch.empty(width, width, device=device)
        torch.nn.init.normal_(W, mean=0.0, std=init_std)
        W.requires_grad_(True)
        weights.append(W)
    return weights


def forward_and_grad_norms(
    weights: list[torch.Tensor],
    activation,
    x: torch.Tensor,
) -> list[float]:
    """Run a forward pass then backward, returning the gradient norm at each layer.

    We use the gradient of the loss w.r.t. the pre-activation *output* of each
    weight matrix (i.e. W @ h_{l-1}) as a proxy for the gradient signal magnitude.
    """
    h = x
    pre_acts = []
    for W in weights:
        z = h @ W.t()       # (batch, width)
        pre_acts.append(z)
        h = activation(z)

    # Scalar loss: mean of final output
    loss = h.mean()
    loss.backward()

    grad_norms = []
    for W in weights:
        if W.grad is not None:
            grad_norms.append(W.grad.norm().item())
        else:
            grad_norms.append(float("nan"))

    # Zero grads for reuse
    for W in weights:
        if W.grad is not None:
            W.grad.zero_()

    return grad_norms


x_input = torch.randn(BATCH, WIDTH, device=device)

# ── Regime 1: Vanishing — sigmoid + small init (std = 0.5) ──────────────────
weights_vanish = build_weights(N_LAYERS, WIDTH, init_std=0.5, device=device)
norms_vanish = forward_and_grad_norms(weights_vanish, torch.sigmoid, x_input)

# ── Regime 2: Exploding — sigmoid + large init (std = 3.0) ──────────────────
weights_explode = build_weights(N_LAYERS, WIDTH, init_std=3.0, device=device)
norms_explode = forward_and_grad_norms(weights_explode, torch.sigmoid, x_input)

# ── Regime 3: Stable — ReLU + He init (std = sqrt(2 / width)) ───────────────
he_std = math.sqrt(2.0 / WIDTH)
weights_stable = build_weights(N_LAYERS, WIDTH, init_std=he_std, device=device)
norms_stable = forward_and_grad_norms(weights_stable, F.relu, x_input)

print("Gradient norms (first 5 layers → last 5 layers):")
print(f"  vanishing  (sigmoid, std=0.5): {[f'{v:.2e}' for v in norms_vanish[:5]]} ... {[f'{v:.2e}' for v in norms_vanish[-5:]]}")
print(f"  exploding  (sigmoid, std=3.0): {[f'{v:.2e}' for v in norms_explode[:5]]} ... {[f'{v:.2e}' for v in norms_explode[-5:]]}")
print(f"  stable     (relu, He):         {[f'{v:.2e}' for v in norms_stable[:5]]} ... {[f'{v:.2e}' for v in norms_stable[-5:]]}")

Gradient norms (first 5 layers → last 5 layers):
  vanishing  (sigmoid, std=0.5): ['7.67e-08', '6.20e-07', '1.53e-06', '2.94e-06', '5.04e-06'] ... ['1.37e-02', '2.48e-02', '4.42e-02', '6.06e-02', '9.48e-02']
  exploding  (sigmoid, std=3.0): ['6.48e+01', '3.89e+01', '2.63e+01', '1.64e+01', '1.16e+01'] ... ['2.61e-02', '2.26e-02', '1.91e-02', '3.01e-02', '2.44e-02']
  stable     (relu, He):         ['2.70e-01', '5.23e-01', '7.28e-01', '9.11e-01', '8.26e-01'] ... ['1.29e+00', '9.84e-01', '1.15e+00', '1.11e+00', '6.85e-01']


## Validation: Manual Gradient Norms Match `torch.autograd`

We confirm that the gradient norms recorded above are consistent with what
`torch.autograd.grad` computes for the same weights and loss.

In [3]:
# Validate: gradient norms from .backward() match torch.autograd.grad for the SAME weights.
# We use a fresh 3-layer stack seeded identically for both paths.

torch.manual_seed(42)
weights_val = build_weights(3, WIDTH, init_std=0.5, device=device)

# ── Path A: torch.autograd.grad ──────────────────────────────────────────────
h_a = x_input
for W in weights_val:
    h_a = torch.sigmoid(h_a @ W.t())
loss_a = h_a.mean()
ag_grads = torch.autograd.grad(loss_a, weights_val)
ag_norms = [g.norm().item() for g in ag_grads]

# Zero any accumulated grads before path B
for W in weights_val:
    if W.grad is not None:
        W.grad.zero_()

# ── Path B: .backward() on the SAME weights ─────────────────────────────────
h_b = x_input
for W in weights_val:
    h_b = torch.sigmoid(h_b @ W.t())
loss_b = h_b.mean()
loss_b.backward()
bwd_norms = [W.grad.norm().item() for W in weights_val]

for i, (ag_n, bwd_n) in enumerate(zip(ag_norms, bwd_norms)):
    diff = abs(ag_n - bwd_n)
    assert diff < 1e-5, f"Layer {i}: autograd norm {ag_n:.6f} vs backward norm {bwd_n:.6f}, diff {diff}"
    print(f"Layer {i+1}: autograd.grad norm = {ag_n:.6f}  backward norm = {bwd_n:.6f}  diff = {diff:.2e}")

print("Validation passed: gradient norms match torch.autograd.grad.")

Layer 1: autograd.grad norm = 0.004963  backward norm = 0.004963  diff = 0.00e+00
Layer 2: autograd.grad norm = 0.044911  backward norm = 0.044911  diff = 0.00e+00
Layer 3: autograd.grad norm = 0.085957  backward norm = 0.085957  diff = 0.00e+00
Validation passed: gradient norms match torch.autograd.grad.


## Idiomatic PyTorch: Deep MLP via `nn.Sequential`

In practice, use `torch.nn` layers and register `backward` hooks to monitor
gradient norms during training without manual weight bookkeeping.

In [4]:
import torch.nn as nn

def make_deep_mlp(n_layers: int, width: int, activation: nn.Module, init_std: float | None = None) -> nn.Sequential:
    """Build a deep MLP and optionally apply custom weight initialisation.

    A fresh activation instance is created per layer via ``copy.deepcopy`` so
    that stateful activations (e.g. ``nn.PReLU``) do not share parameters across
    layers — idiomatic PyTorch practice.
    """
    import copy
    layers: list[nn.Module] = []
    for _ in range(n_layers):
        linear = nn.Linear(width, width, bias=False)
        if init_std is not None:
            nn.init.normal_(linear.weight, std=init_std)
        layers.append(linear)
        layers.append(copy.deepcopy(activation))
    return nn.Sequential(*layers)


def measure_grad_norms_nn(model: nn.Sequential, x: torch.Tensor) -> list[float]:
    """Forward + backward, return per-Linear-layer weight gradient norms."""
    loss = model(x).mean()
    loss.backward()
    norms = []
    for m in model.modules():
        if isinstance(m, nn.Linear) and m.weight.grad is not None:
            norms.append(m.weight.grad.norm().item())
    model.zero_grad()
    return norms


torch.manual_seed(0)
he_std = math.sqrt(2.0 / WIDTH)

mlp_vanish  = make_deep_mlp(N_LAYERS, WIDTH, nn.Sigmoid(), init_std=0.5).to(device)
mlp_explode = make_deep_mlp(N_LAYERS, WIDTH, nn.Sigmoid(), init_std=3.0).to(device)
mlp_stable  = make_deep_mlp(N_LAYERS, WIDTH, nn.ReLU(),    init_std=he_std).to(device)

nn_norms_vanish  = measure_grad_norms_nn(mlp_vanish,  x_input)
nn_norms_explode = measure_grad_norms_nn(mlp_explode, x_input)
nn_norms_stable  = measure_grad_norms_nn(mlp_stable,  x_input)

print("nn.Sequential gradient norms (first 3 / last 3 layers):")
print(f"  vanishing : {nn_norms_vanish[:3]}  ...  {nn_norms_vanish[-3:]}")
print(f"  exploding : {nn_norms_explode[:3]}  ...  {nn_norms_explode[-3:]}")
print(f"  stable    : {nn_norms_stable[:3]}  ...  {nn_norms_stable[-3:]}")

nn.Sequential gradient norms (first 3 / last 3 layers):
  vanishing : [5.326244831849181e-07, 5.005689672543667e-06, 1.0620515240589157e-05]  ...  [0.04668523371219635, 0.055672064423561096, 0.08979032188653946]
  exploding : [124.67477416992188, 75.86402893066406, 38.27120590209961]  ...  [0.023473205044865608, 0.018996024504303932, 0.02344803512096405]
  stable    : [0.4234008193016052, 1.1145271062850952, 1.5677343606948853]  ...  [1.8114012479782104, 1.576418399810791, 1.3008977174758911]


## Gradient Clipping for the Exploding Case

`torch.nn.utils.clip_grad_norm_` rescales all gradients so their global L2 norm
does not exceed a threshold. This bounds the update magnitude without zeroing
gradients entirely.

In [5]:
CLIP_THRESHOLD = 1.0

torch.manual_seed(0)
mlp_clip = make_deep_mlp(N_LAYERS, WIDTH, nn.Sigmoid(), init_std=3.0).to(device)

# ── Measure gradient norm BEFORE clipping ───────────────────────────────────
loss_before = mlp_clip(x_input).mean()
loss_before.backward()
# clip_grad_norm_ clips *in-place* and returns the total pre-clip norm.
pre_clip_norm = torch.nn.utils.clip_grad_norm_(mlp_clip.parameters(), max_norm=float("inf"))
print(f"Global gradient norm BEFORE clipping : {pre_clip_norm:.4f}")
mlp_clip.zero_grad()

# ── Apply gradient clipping (max_norm = CLIP_THRESHOLD) ─────────────────────
loss_after = mlp_clip(x_input).mean()
loss_after.backward()
# Returns total pre-clip norm; grads are now rescaled in place.
returned_norm = torch.nn.utils.clip_grad_norm_(mlp_clip.parameters(), max_norm=CLIP_THRESHOLD)
# Compute the post-clip norm by measuring the gradients directly.
post_clip_norm = sum(p.grad.norm() ** 2 for p in mlp_clip.parameters() if p.grad is not None).sqrt().item()

print(f"Global gradient norm returned by clip : {returned_norm:.4f}  (pre-clip total)")
print(f"Global gradient norm AFTER  clipping  : {post_clip_norm:.4f}")
print(f"Clipping threshold                    : {CLIP_THRESHOLD}")

assert post_clip_norm <= CLIP_THRESHOLD + 1e-4, (
    f"Clipping failed: post-clip norm = {post_clip_norm:.6f} > threshold = {CLIP_THRESHOLD}"
)
print("Gradient clipping assertion passed.")

Global gradient norm BEFORE clipping : 207.0813
Global gradient norm returned by clip : 207.0813  (pre-clip total)
Global gradient norm AFTER  clipping  : 1.0000
Clipping threshold                    : 1.0
Gradient clipping assertion passed.


## Plots: Gradient Norm vs. Layer Depth

We plot gradient norms for all three regimes from the manual implementation
(which gives us per-layer control).  The x-axis is layer index (0 = input side);
lower indices receive the gradient *last* in backpropagation.

In [6]:
layers = list(range(1, N_LAYERS + 1))

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, norms, title, color in [
    (axes[0], norms_vanish,  "Vanishing (sigmoid, std=0.5)", "tab:blue"),
    (axes[1], norms_explode, "Exploding (sigmoid, std=3.0)", "tab:red"),
    (axes[2], norms_stable,  "Stable (ReLU, He init)",       "tab:green"),
]:
    ax.plot(layers, norms, marker="o", markersize=4, lw=1.5, color=color)
    ax.set_xlabel("Layer index (1 = closest to input)")
    ax.set_ylabel("||∇W|| (gradient norm)")
    ax.set_title(title)
    ax.set_yscale("log")
    ax.grid(True, which="both", ls="--", alpha=0.5)

fig.suptitle("Per-layer Gradient Norms vs. Depth", fontsize=13)
plt.tight_layout()
plt.savefig("gradient_norms_plot.png", dpi=100, bbox_inches="tight")
plt.close()
print("Plot saved to gradient_norms_plot.png")

Plot saved to gradient_norms_plot.png


## Discussion

### Vanishing gradients (sigmoid + small init)

With `std = 0.5`, the weight matrices have small singular values. The sigmoid
derivative is bounded by `0.25`, so each layer multiplies the gradient by at most
`0.25 × ||W_i||`. Over 25 layers this product approaches **zero** exponentially.
Early layers (low index) receive near-zero gradient and effectively stop learning.

### Exploding gradients (sigmoid + large init)

With `std = 3.0`, weight singular values are large. Even though sigmoid derivatives
are bounded, the weight norm dominates and the product **grows** exponentially.
Updates become numerically unstable; loss goes to `nan` in practice.

### Stable training (ReLU + He init)

**He initialisation** sets `std = sqrt(2 / fan_in)`. For an active ReLU unit
the derivative is exactly 1, and He init keeps the activation variance constant
through layers, so the Jacobian norm stays near 1. Gradient norms remain roughly
constant from output to input — every layer receives a useful signal.

### Gradient clipping

`clip_grad_norm_` projects the gradient vector onto a ball of radius `max_norm`.
This is a pragmatic safeguard: it does not prevent exploding gradients from
occurring, but it prevents a single bad batch from causing a catastrophic update.
It is standard in recurrent networks (LSTMs) and transformer training.

### Mitigations (summary)

| Problem | Mitigation | Mechanism |
|---|---|---|
| Vanishing | ReLU / He init | Gradient = 1 on active paths, unit variance |
| Vanishing | Batch Normalisation (`[[batch-normalization]]`) | Re-centres activations |
| Vanishing | Residual connections | Shortcut gradient paths bypass saturation |
| Exploding | Gradient clipping (training) | Bounds update magnitude during training; does not fix init |
| Exploding | Weight regularisation | Penalises large singular values |
| Vanishing | Xavier init (tanh/sigmoid) | Targets unit-variance activations at init for saturating activations |
| Vanishing | He/Kaiming init (ReLU) | Accounts for ~50% inactive units; prevents vanishing at init for ReLU-family |

## Takeaways

1. **Vanishing gradients** arise when per-layer Jacobian norms are consistently
   below 1 — common with sigmoid/tanh activations and small initialisations.
   Early layers stop receiving useful gradient signal.
2. **Exploding gradients** arise when Jacobian norms are consistently above 1 —
   common with large initialisations. Updates become dominated by enormous steps
   and training diverges.
3. **ReLU + He init** keeps gradient norms approximately constant across depth
   because: (a) the derivative is 1 on active paths, and (b) He init targets
   unit output variance accounting for the ~50% inactive fraction.
4. **Gradient clipping** (`clip_grad_norm_`) is a practical safeguard that bounds
   the global gradient L2 norm, preventing catastrophic updates in the exploding
   regime.
5. The primary diagnostic is **gradient norm by layer**, not just training loss —
   loss can appear normal even while early layers are starved of signal.
6. Cross-links: `[[backpropagation]]` explains the chain-rule mechanics;
   `[[batch-normalization]]` provides complementary stabilisation at the
   activation-distribution level.